# 03 Statistical Testing and Regression

This notebook keeps the testing and regression code close to the assignment style while adding clearer explanations.


## Import Libraries and Load Data


In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats import ttest_ind
import statsmodels.api as sm


In [2]:
cleaned_path = '../data/processed/cleaned_luminatech_transactions.csv'
sample_path = '../data/sample/sample_luminatech_data.csv'

if os.path.exists(cleaned_path):
    Dataset_cleaned = pd.read_csv(cleaned_path, dtype={'customer_code': str}, low_memory=False)
else:
    Dataset_cleaned = pd.read_csv(sample_path, dtype={'customer_code': str})

for col in ['invoice_date', 'order_date']:
    Dataset_cleaned[col] = pd.to_datetime(Dataset_cleaned[col], errors='coerce')

if 'profit' not in Dataset_cleaned.columns:
    Dataset_cleaned['profit'] = Dataset_cleaned['value_sales'] - Dataset_cleaned['value_cost']

Dataset_cleaned['profit_margin'] = (Dataset_cleaned['value_sales'] - Dataset_cleaned['value_cost']) / Dataset_cleaned['value_sales']
Dataset_cleaned['profit_margin'] = Dataset_cleaned['profit_margin'].replace([np.inf, -np.inf], np.nan)
Dataset_cleaned = Dataset_cleaned.dropna(subset=['profit_margin'])

Dataset_cleaned.head()


,accounting_date,fiscal_year,fiscal_month,calendar_year,calendar_month,calendar_day,company_code,customer_code,customer_district_code,item_code,...,currency,invoice_number,line_number,invoice_date,customer_order_number,order_date,dss_update_time,profit,time_gap,profit_margin
0,2012-05-09,2012,11,2012,5,9,101,411800601,410,GENIE8WWWBC,...,AUD,2217887,1,2012-05-09,2865354,2012-05-09,49:58.7,40.2024,0,0.184077
1,2012-02-16,2012,8,2012,2,16,101,361000403,300,GENIE8WWWBC,...,AUD,2185745,1,2012-02-16,2833515,2012-02-16,49:58.7,12.8232,0,0.334984
2,2012-05-09,2012,11,2012,5,9,101,361000403,300,GENIE8WWWBC,...,AUD,2217807,1,2012-05-09,2864857,2012-05-08,49:58.7,14.7432,1,0.366746
3,2012-05-18,2012,11,2012,5,18,101,565540415,500,GENIE8WWWBC,...,AUD,2222758,1,2012-05-18,2869759,2012-05-18,49:58.7,7.3716,0,0.366746
4,2012-01-09,2012,7,2012,1,9,101,565540415,500,GENIE8WWWBC,...,AUD,2170374,1,2012-01-09,2819189,2012-01-09,49:58.7,6.4116,0,0.334984


## Test 1: New vs Returning Customer Spend

This checks whether first-purchase spending is different from returning-customer spending.


In [3]:
Dataset_cleaned['first_purchase'] = Dataset_cleaned.groupby('customer_code')['invoice_date'].transform('min')
Dataset_cleaned['customer_type'] = Dataset_cleaned.apply(
    lambda row: 'New' if row['invoice_date'] == row['first_purchase'] else 'Returning',
    axis=1
)

monthly_spend = Dataset_cleaned.groupby(['customer_type', 'invoice_date'])['value_sales'].mean().reset_index()

new_customers_spend = monthly_spend[monthly_spend['customer_type'] == 'New']['value_sales']
returning_customers_spend = monthly_spend[monthly_spend['customer_type'] == 'Returning']['value_sales']

t_stat, p_value = ttest_ind(new_customers_spend, returning_customers_spend, equal_var=False, nan_policy='omit')

print('Two-sample t-test Statistic:', t_stat)
print('p-value:', p_value)


Two-sample t-test Statistic: 6.215915304478948
p-value: 1.002821131673738e-09


## Test 2: Profit Margin Comparison Between Two Districts

The original assignment compared District 200 and District 300 to understand whether profit margins differ between two important regions.


In [4]:
district_a_profit = Dataset_cleaned[Dataset_cleaned['customer_district_code'] == 200]['profit_margin']
district_b_profit = Dataset_cleaned[Dataset_cleaned['customer_district_code'] == 300]['profit_margin']

if len(district_a_profit) > 0 and len(district_b_profit) > 0:
    t_stat, p_value = ttest_ind(district_a_profit, district_b_profit, equal_var=False, nan_policy='omit')
    print('Two-sample t-test Statistic:', t_stat)
    print('p-value:', p_value)
else:
    print('One or both district samples are empty in this dataset.')


Two-sample t-test Statistic: 5.4939507670246375
p-value: 3.932060650448282e-08


## Regression 1: Quantity, Cost, and Market Segment Influence on Sales

This regression checks how sales value changes with quantity, cost, and market segment.


In [5]:
regression_data = Dataset_cleaned[['value_sales', 'market_segment', 'value_quantity', 'value_cost']].dropna()
regression_data = pd.get_dummies(regression_data, columns=['market_segment'], drop_first=True)

X = regression_data.drop('value_sales', axis=1)
X = sm.add_constant(X).astype(float)
y = regression_data['value_sales'].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            value_sales   R-squared:                       0.806
Model:                            OLS   Adj. R-squared:                  0.806
Method:                 Least Squares   F-statistic:                 4.076e+06
Date:                Sun, 03 May 2026   Prob (F-statistic):               0.00
Time:                        00:04:28   Log-Likelihood:            -1.6888e+07
No. Observations:             1966082   AIC:                         3.378e+07
Df Residuals:                 1966079   BIC:                         3.378e+07
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const             77.4803      0.937     82.

## Regression 2: Cost and Invoice Month Influence on Sales

This regression adds invoice month as a simple seasonality variable.


In [6]:
Dataset_cleaned['invoice_month'] = Dataset_cleaned['invoice_date'].dt.month

seasonality_data = Dataset_cleaned[['value_sales', 'value_cost', 'invoice_month']].dropna()

X = seasonality_data[['value_cost', 'invoice_month']]
X = sm.add_constant(X).astype(float)
y = seasonality_data['value_sales'].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            value_sales   R-squared:                       0.805
Model:                            OLS   Adj. R-squared:                  0.805
Method:                 Least Squares   F-statistic:                 4.056e+06
Date:                Sun, 03 May 2026   Prob (F-statistic):               0.00
Time:                        00:04:29   Log-Likelihood:            -1.6892e+07
No. Observations:             1966082   AIC:                         3.378e+07
Df Residuals:                 1966079   BIC:                         3.378e+07
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            79.3389      2.054     38.622

## Interpretation Note

These regression models help explain relationships in the historical data. They should not be treated as proof of causation without more business context and stronger validation.
